# Bulk Feature Extraction
Extract features at one time.

In [1]:
import sys
from pathlib import Path
import torch

REPO_ROOT = Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from config import PROJECT_ROOT, RANDOM_SEED
from data_split import create_train_val_split
from extract_egemaps_feature import extract_egemaps_features_from_csv
from extract_XLSR_feature import extract_features_from_csv
from model import SSLModel

DEVICE = ("cuda" if torch.cuda.is_available() else
          ("mps" if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available() else "cpu"))
FREEZE_XLSR = True
SPLIT_RANDOM_SEED = RANDOM_SEED

/opt/anaconda3/envs/madress-2023-x86/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATASETS = {
    "Pitt": {
        "raw_audio_dir": PROJECT_ROOT / "data/raw/Pitt",
        "egemaps_csv": [
            PROJECT_ROOT / "data/processed/Pitt-egemaps-train.csv",
            PROJECT_ROOT / "data/processed/Pitt-egemaps-val.csv",
        ],
        "egemaps_feature_dir": PROJECT_ROOT / "data/processed/Pitt_egemaps_features",
        "xlsr_csv": [
            PROJECT_ROOT / "data/processed/Pitt-xlsr-train.csv",
            PROJECT_ROOT / "data/processed/Pitt-xlsr-val.csv",
        ],
        "xlsr_feature_dir": PROJECT_ROOT / "data/processed/Pitt_xlsr_features",
    },
    "ADReSS": {
        "raw_audio_dir": PROJECT_ROOT / "data/raw/ADReSS",
        "egemaps_csv": [
            PROJECT_ROOT / "data/processed/ADReSS-egemaps-train.csv",
            PROJECT_ROOT / "data/processed/ADReSS-egemaps-val.csv",
        ],
        "egemaps_feature_dir": PROJECT_ROOT / "data/processed/ADReSS_egemaps_features",
        "xlsr_csv": [
            PROJECT_ROOT / "data/processed/ADReSS-xlsr-train.csv",
            PROJECT_ROOT / "data/processed/ADReSS-xlsr-val.csv",
        ],
        "xlsr_feature_dir": PROJECT_ROOT / "data/processed/ADReSS_xlsr_features",
    },
    "Lu": {
        "raw_audio_dir": PROJECT_ROOT / "data/raw/Lu",
        "egemaps_csv": [
            PROJECT_ROOT / "data/processed/Lu-egemaps-train.csv",
            PROJECT_ROOT / "data/processed/Lu-egemaps-val.csv",
        ],
        "egemaps_feature_dir": PROJECT_ROOT / "data/processed/Lu_egemaps_features",
        "xlsr_csv": [
            PROJECT_ROOT / "data/processed/Lu-xlsr-train.csv",
            PROJECT_ROOT / "data/processed/Lu-xlsr-val.csv",
        ],
        "xlsr_feature_dir": PROJECT_ROOT / "data/processed/Lu_xlsr_features",
    }
}

SELECTED_DATASETS = ["Pitt", "ADReSS", "Lu"]
RUN_EGEMAP = True
RUN_XLSR = True


In [3]:
def ensure_csv_pair(dataset_name, csv_list, feature_dir, raw_audio_dir, *, is_xlsr):
    train_csv, val_csv = csv_list
    if not (train_csv.exists() and val_csv.exists()):
        create_train_val_split(
            raw_audio_dir=raw_audio_dir,
            train_csv_path=train_csv,
            val_csv_path=val_csv,
            feature_dir_name=feature_dir,
            random_seed=SPLIT_RANDOM_SEED,
            dataset_name=dataset_name,
            xlsr=is_xlsr,
        )
    return [train_csv, val_csv]


def run_egemaps_extraction(dataset_name, cfg):
    if not RUN_EGEMAP:
        return
    csv_list = ensure_csv_pair(
        dataset_name,
        cfg["egemaps_csv"],
        cfg["egemaps_feature_dir"],
        cfg["raw_audio_dir"],
        is_xlsr=False,
    )
    for csv_path in csv_list:
        print(f"[{dataset_name}] eGeMAP {csv_path.stem}")
        extract_egemaps_features_from_csv(csv_path, cfg["raw_audio_dir"])


def run_xlsr_extraction(dataset_name, cfg):
    if not RUN_XLSR:
        return
    csv_list = ensure_csv_pair(
        dataset_name,
        cfg["xlsr_csv"],
        cfg["xlsr_feature_dir"],
        cfg["raw_audio_dir"],
        is_xlsr=True,
    )
    feature_dir = cfg["xlsr_feature_dir"]
    feature_dir.mkdir(parents=True, exist_ok=True)
    ssl_model = SSLModel(device=DEVICE, freeze_xlsr=FREEZE_XLSR)
    for csv_path in csv_list:
        print(f"[{dataset_name}] XLSR {csv_path.stem}")
        extract_features_from_csv(
            csv_path=csv_path,
            split_name=f"{dataset_name}-{csv_path.stem}",
            raw_audio_dir=cfg["raw_audio_dir"],
            xlsr_features_dir=feature_dir,
            device=DEVICE,
            ssl_model=ssl_model,
            freeze_xlsr=FREEZE_XLSR,
        )


for dataset_name in SELECTED_DATASETS:
    cfg = DATASETS[dataset_name]
    print(f"\n==================== {dataset_name} ====================")
    run_egemaps_extraction(dataset_name, cfg)
    run_xlsr_extraction(dataset_name, cfg)


==================== Pitt ====================
[Pitt] eGeMAP Pitt-egemaps-train

============= Extraction eGeMaps features =============
440 Audio Files


Extracting: 100%|██████████| 440/440 [00:00<00:00, 53288.69it/s]


Successfully extracted: 0
Already Exists (Skipped): 440
Total: 440
Errors: 0
[Pitt] eGeMAP Pitt-egemaps-val

============= Extraction eGeMaps features =============
111 Audio Files


Extracting: 100%|██████████| 111/111 [00:00<00:00, 41609.41it/s]

Successfully extracted: 0
Already Exists (Skipped): 111
Total: 111
Errors: 0
XLSR:Using original XLSR model


...

[Lu] XLSR Lu-xlsr-train

============= Extracting XLSR features for Lu-Lu-xlsr-train =============
42 Audio Files


Extracting Lu-Lu-xlsr-train: 100%|██████████| 42/42 [00:00<00:00, 38564.09it/s]


Successfully extracted: 0
Already exists (skipped): 42
Errors: 0
Total: 42
[Lu] XLSR Lu-xlsr-val

============= Extracting XLSR features for Lu-Lu-xlsr-val =============
11 Audio Files


Extracting Lu-Lu-xlsr-val: 100%|██████████| 11/11 [00:00<00:00, 15127.00it/s]

Successfully extracted: 0
Already exists (skipped): 11
Errors: 0
Total: 11
